# Data Lifecycle & Retention Audit Notebook

Portable audit of Time Travel retention, storage lifecycle (active / time travel / fail-safe /
clone-retained), and data staleness — directly mapping to the JD's "develop and implement
enterprise data retention and archival strategies... reduce operational risk and storage costs
associated with outdated or inactive data." Run top to bottom. It surfaces:

1. **Time Travel retention settings** — at account, database, schema, and table level, and where they diverge from what you'd expect
2. **Storage breakdown by lifecycle stage** — active vs. time travel vs. fail-safe vs. clone-retained bytes, which is where retention settings actually show up as cost
3. **Stale / inactive tables** — large tables nobody has queried in months, ranked by storage cost
4. **Transient vs. permanent table review** — permanent tables that look like staging/scratch data paying for fail-safe they don't need
5. **Clone proliferation** — storage held down by clones, which is easy to lose track of
6. **A rolled-up recommendation summary**

### Prerequisites
- A role with `IMPORTED PRIVILEGES` on `SNOWFLAKE` (or `ACCOUNTADMIN`).
- Nothing here is destructive. Every cell reads.

### Why this matters beyond cost
Retention isn't only a storage-cost question in a regulated environment — data kept *longer*
than policy requires is itself a compliance exposure, and data with *no* Time Travel window
before an intended retention period is a recovery risk. Get the actual retention policy
requirements from compliance/legal before treating "shorter is better" as the answer; the
right number is a policy decision, not a technical one.


In [ ]:
-- ============================================================
-- PARAMETERS
-- ============================================================
SET stale_days = 180;          -- no query activity in this many days = "stale" candidate
SET large_table_gb = 10;       -- only rank stale/lifecycle findings above this size as priority

SELECT $stale_days AS stale_days, $large_table_gb AS large_table_gb;


## 1. Time Travel retention settings

The account-level default sets the floor for anything not explicitly overridden. Table-level
`RETENTION_TIME` shows what's actually in effect per object — including any overrides that
predate the current admin and may no longer be intentional.


In [ ]:
-- Account, database, and schema level defaults
SHOW PARAMETERS LIKE 'DATA_RETENTION_TIME_IN_DAYS' IN ACCOUNT;


In [ ]:
-- Effective retention per table, plus whether it's transient (transient tables skip Fail-safe)
SELECT
    table_catalog AS database_name,
    table_schema,
    table_name,
    table_type,
    is_transient,
    retention_time AS retention_days,
    ROUND(bytes / POWER(1024,3), 2) AS gb,
    row_count,
    last_altered
FROM snowflake.account_usage.tables
WHERE deleted IS NULL
  AND table_schema != 'INFORMATION_SCHEMA'
ORDER BY retention_time DESC, gb DESC
LIMIT 300;


In [ ]:
-- Retention distribution — how many tables at each setting, and how much storage they represent
SELECT
    retention_time AS retention_days,
    is_transient,
    COUNT(*)                                   AS table_count,
    ROUND(SUM(bytes) / POWER(1024,3), 1)       AS total_gb
FROM snowflake.account_usage.tables
WHERE deleted IS NULL
  AND table_schema != 'INFORMATION_SCHEMA'
GROUP BY retention_time, is_transient
ORDER BY retention_days DESC;


**Watch for:** tables at the maximum retention (90 days, Enterprise edition+) that hold
routine/high-churn data — that's the most expensive Time Travel configuration for data that
probably doesn't need a 90-day recovery window. Also watch for `retention_days = 0` on
anything that isn't explicitly meant to skip Time Travel entirely, since that removes the
undo-a-mistake safety net along with the storage cost.


## 2. Storage breakdown by lifecycle stage

`TABLE_STORAGE_METRICS` is where retention settings actually show up as billed storage —
active data, Time Travel, Fail-safe, and storage retained solely because a clone depends on
it, broken out separately.


In [ ]:
-- Storage by lifecycle stage, account-wide totals
SELECT
    ROUND(SUM(active_bytes) / POWER(1024,3), 1)              AS active_gb,
    ROUND(SUM(time_travel_bytes) / POWER(1024,3), 1)         AS time_travel_gb,
    ROUND(SUM(failsafe_bytes) / POWER(1024,3), 1)            AS failsafe_gb,
    ROUND(SUM(retained_for_clone_bytes) / POWER(1024,3), 1)  AS clone_retained_gb,
    ROUND(SUM(active_bytes + time_travel_bytes + failsafe_bytes + retained_for_clone_bytes) / POWER(1024,3), 1) AS total_gb
FROM snowflake.account_usage.table_storage_metrics
WHERE deleted IS NULL;


In [ ]:
-- Tables where Time Travel + Fail-safe storage is large relative to active data —
-- a high ratio here usually means high write/delete/update churn on a table with a longer
-- retention window than that churn pattern needs.
SELECT
    table_catalog AS database_name,
    table_schema,
    table_name,
    ROUND(active_bytes / POWER(1024,3), 2)      AS active_gb,
    ROUND(time_travel_bytes / POWER(1024,3), 2) AS time_travel_gb,
    ROUND(failsafe_bytes / POWER(1024,3), 2)    AS failsafe_gb,
    ROUND(DIV0(time_travel_bytes + failsafe_bytes, active_bytes), 2) AS overhead_ratio
FROM snowflake.account_usage.table_storage_metrics
WHERE deleted IS NULL
  AND active_bytes > ($large_table_gb * POWER(1024,3))
ORDER BY overhead_ratio DESC
LIMIT 100;


## 3. Stale / inactive tables

Tables with no read or write activity in `stale_days` are the clearest archival/deletion
candidates — ranked by size so the highest storage-cost stale tables surface first.


In [ ]:
-- Tables with no query activity in the stale window, ranked by size
WITH recent_activity AS (
    SELECT DISTINCT
        base.value:"objectName"::string AS object_name
    FROM snowflake.account_usage.access_history,
         LATERAL FLATTEN(input => base_objects_accessed) base
    WHERE query_start_time >= DATEADD(day, -$stale_days, CURRENT_TIMESTAMP())
)
SELECT
    t.table_catalog AS database_name,
    t.table_schema,
    t.table_name,
    ROUND(t.bytes / POWER(1024,3), 2) AS gb,
    t.row_count,
    t.last_altered,
    t.table_owner
FROM snowflake.account_usage.tables t
WHERE t.deleted IS NULL
  AND t.table_schema != 'INFORMATION_SCHEMA'
  AND t.bytes > (1 * POWER(1024,3))  -- ignore trivially small tables
  AND NOT EXISTS (
        SELECT 1 FROM recent_activity ra
        WHERE ra.object_name = t.table_catalog || '.' || t.table_schema || '.' || t.table_name
      )
ORDER BY gb DESC
LIMIT 200;


This uses `ACCESS_HISTORY`, which is more reliable than `QUERY_HISTORY` text-matching for
"was this table actually read," but check its retention window in this account (commonly
around a year) — a table absent from `ACCESS_HISTORY` because it predates that window isn't
necessarily unused, just unmeasured. Cross-check anything borderline against `LAST_ALTERED`
before recommending removal.


## 4. Transient vs. permanent table review

Transient tables skip Fail-safe entirely (no 7-day disaster-recovery storage charge), which
is the right choice for staging/scratch/intermediate data that can simply be reloaded or
rebuilt. Permanent tables that look like staging data by name or by their downstream usage
pattern are worth converting.


In [ ]:
-- Large permanent tables with staging/scratch-like naming, still paying for Fail-safe
SELECT
    table_catalog AS database_name,
    table_schema,
    table_name,
    ROUND(bytes / POWER(1024,3), 2) AS gb,
    is_transient,
    retention_time,
    last_altered
FROM snowflake.account_usage.tables
WHERE deleted IS NULL
  AND table_schema != 'INFORMATION_SCHEMA'
  AND is_transient = 'NO'
  AND bytes > (1 * POWER(1024,3))
  AND (table_name ILIKE '%STG%' OR table_name ILIKE '%STAGING%' OR table_name ILIKE '%TMP%'
       OR table_name ILIKE '%TEMP%' OR table_name ILIKE '%SCRATCH%' OR table_schema ILIKE '%STAGING%'
       OR table_schema ILIKE '%STG%')
ORDER BY gb DESC
LIMIT 100;


## 5. Clone proliferation

`RETAINED_FOR_CLONE_BYTES` in Section 2 shows storage kept alive purely because a clone still
references it — this drills into which source tables are behind the largest amount of
clone-retained storage, since that's easy to accumulate silently (e.g., a "quick test clone"
from months ago that nobody dropped).


In [ ]:
-- Tables with the most storage retained on their behalf due to dependent clones
SELECT
    table_catalog AS database_name,
    table_schema,
    table_name,
    ROUND(retained_for_clone_bytes / POWER(1024,3), 2) AS clone_retained_gb,
    ROUND(active_bytes / POWER(1024,3), 2)             AS active_gb,
    last_altered
FROM snowflake.account_usage.table_storage_metrics
WHERE deleted IS NULL
  AND retained_for_clone_bytes > (1 * POWER(1024,3))
ORDER BY clone_retained_gb DESC
LIMIT 100;


## 6. Rolled-up recommendation summary


In [ ]:
-- Per-database summary combining lifecycle storage and stale-table counts
WITH storage AS (
    SELECT
        table_catalog AS database_name,
        ROUND(SUM(active_bytes) / POWER(1024,3), 1)              AS active_gb,
        ROUND(SUM(time_travel_bytes) / POWER(1024,3), 1)         AS time_travel_gb,
        ROUND(SUM(failsafe_bytes) / POWER(1024,3), 1)            AS failsafe_gb,
        ROUND(SUM(retained_for_clone_bytes) / POWER(1024,3), 1)  AS clone_retained_gb
    FROM snowflake.account_usage.table_storage_metrics
    WHERE deleted IS NULL
    GROUP BY table_catalog
),
recent_activity AS (
    SELECT DISTINCT base.value:"objectName"::string AS object_name
    FROM snowflake.account_usage.access_history,
         LATERAL FLATTEN(input => base_objects_accessed) base
    WHERE query_start_time >= DATEADD(day, -$stale_days, CURRENT_TIMESTAMP())
),
stale AS (
    SELECT
        t.table_catalog AS database_name,
        COUNT(*) AS stale_table_count,
        ROUND(SUM(t.bytes) / POWER(1024,3), 1) AS stale_gb
    FROM snowflake.account_usage.tables t
    WHERE t.deleted IS NULL
      AND t.table_schema != 'INFORMATION_SCHEMA'
      AND t.bytes > (1 * POWER(1024,3))
      AND NOT EXISTS (
            SELECT 1 FROM recent_activity ra
            WHERE ra.object_name = t.table_catalog || '.' || t.table_schema || '.' || t.table_name
          )
    GROUP BY t.table_catalog
)
SELECT
    s.database_name,
    s.active_gb,
    s.time_travel_gb,
    s.failsafe_gb,
    s.clone_retained_gb,
    COALESCE(st.stale_table_count, 0) AS stale_table_count,
    COALESCE(st.stale_gb, 0)          AS stale_gb,
    ARRAY_TO_STRING(ARRAY_CONSTRUCT_COMPACT(
        IFF(s.clone_retained_gb > 5, 'REVIEW CLONE RETENTION', NULL),
        IFF(COALESCE(st.stale_gb, 0) > 5, 'REVIEW STALE TABLES FOR ARCHIVAL', NULL),
        IFF(DIV0(s.time_travel_gb + s.failsafe_gb, NULLIF(s.active_gb,0)) > 0.5, 'REVIEW RETENTION SETTINGS (high TT/FS overhead)', NULL)
    ), '; ') AS recommendations
FROM storage s
LEFT JOIN stale st ON s.database_name = st.database_name
ORDER BY s.active_gb + s.time_travel_gb + s.failsafe_gb + s.clone_retained_gb DESC;


### Notes, caveats, and next steps

- **Get the actual retention policy before recommending changes.** "This table has a 90-day
  Time Travel window" is a finding; "it should have a 7-day window" is a policy decision that
  belongs to compliance/legal in a healthcare environment, not a default technical opinion.
- **`ACCESS_HISTORY` retention is finite.** Confirm how far back it goes in this account before
  treating Section 3's "stale" list as complete — a table with no visible activity might simply
  predate the tracked window.
- **Pair with the RBAC audit notebook's individually-owned-objects section** — a stale table
  owned by a person rather than a role is a strong double-signal for cleanup.
- **Archival, not just deletion, is often the right answer** for regulated data — moving cold
  data to a cheaper storage tier or an archive database with tighter access controls can
  satisfy both the cost and the retention-policy goals at once.
